## Lab 01 – Snowflake Notebook Fundamentals
**Snowflake Fundamentals 4-day class Lab · © 2026 Innovation In Software Corporation. All rights reserved.**

Topics covered:
1. Snowflake Notebooks — SQL and Python cells side by side
2. Session context — connecting to Snowflake via `get_active_session()`
3. SHOW ACCOUNTS — listing account details with SQL and Snowpark
4. DataFrame operations with Pandas
5. Writing a DataFrame back to Snowflake as a temporary table
6. UNPIVOT — transforming wide rows into key-value pairs

---
## PART 1 — INSTRUCTOR DEMO
> Students follow along. No typing required until Part 2.

### DEMO 1 | Context Setup

> **[INSTRUCTOR NOTE]**
> Every Snowflake Notebook session must specify which database to use before running SQL. The cell language is set per-cell using the cell type marker — `#%% sql` for SQL, `#%% python` for Python.

In [ ]:
CREATE DATABASE IF NOT EXISTS demo_db;
USE DATABASE demo_db;
USE SCHEMA public;


### DEMO 2 | SHOW ACCOUNTS — SQL Cell

> **[INSTRUCTOR NOTE]**
> `SHOW ACCOUNTS` is a metadata command requiring ACCOUNTADMIN or ORGADMIN. It returns account name, locator, URL, and region without touching any warehouse (zero compute cost).

In [ ]:
%%sql -r dataframe_1
SHOW ACCOUNTS;

### DEMO 3 | Snowpark Session and DataFrame

> **[INSTRUCTOR NOTE]**
> `get_active_session()` retrieves the live Snowflake session inside the notebook. This is the entry point for all Snowpark DataFrame operations. The session object is reused throughout the notebook.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()

df_sp = session.sql("SHOW ACCOUNTS")
df = pd.DataFrame(df_sp.collect())

df

### DEMO 4 | Enriching a DataFrame in Python

> **[INSTRUCTOR NOTE]**
> Python cells operate on Pandas DataFrames returned from Snowpark `.collect()`. A derived column (`snowsight_url`) is added by string concatenation — this runs entirely in the notebook Python runtime, not in Snowflake.

In [ ]:
df["snowsight_url"] = (
    "https://app.snowflake.com/"
    + df["organization_name"].str.lower()
    + "/"
    + df["account_name"].str.lower()
)

fields = [
    "organization_name", "account_name", "account_locator",
    "account_url", "account_locator_url", "snowsight_url"
]

for i, row in df.iterrows():
    for field in fields:
        print(f"{field:22}: {row[field]}")

### DEMO 5 | Write DataFrame to a Temporary Table

> **[INSTRUCTOR NOTE]**
> `save_as_table(..., table_type='temporary')` writes the Pandas-enriched DataFrame back to Snowflake as a session-scoped temporary table. Temporary tables are invisible to other sessions and are dropped automatically when the session ends.

In [ ]:
snow_df = session.create_dataframe(df)
snow_df.write.mode("overwrite").save_as_table("TEMP_ACCOUNTS", table_type="temporary")

### DEMO 6 | Query the Temporary Table from SQL

> **[INSTRUCTOR NOTE]**
> SQL cells and Python cells share the same Snowflake session — a temporary table written by a Python cell is immediately queryable in a SQL cell within the same notebook run.

In [ ]:
SELECT * FROM TEMP_ACCOUNTS;

### DEMO 7 | UNPIVOT — Wide Row to Key-Value Pairs

> **[INSTRUCTOR NOTE]**
> UNPIVOT rotates selected columns into rows, producing a key-value representation. The derived `snowsight_url` column is added inline in the subquery before UNPIVOT.

In [ ]:
%%sql -r dataframe_4
SELECT *
FROM (
    SELECT
        "organization_name",
        "account_name",
        "account_locator",
        "account_url",
        "account_locator_url",
        'https://app.snowflake.com/'
            || LOWER("organization_name") || '/'
            || LOWER("account_name") AS "snowsight_url"
    FROM TEMP_ACCOUNTS
)
UNPIVOT (
    value FOR key IN (
        "organization_name",
        "account_name",
        "account_locator",
        "account_url",
        "account_locator_url",
        "snowsight_url"
    )
);

---
## PART 2 — STUDENT EXERCISES
> Complete each exercise. Run each cell and verify the result.
> All exercises are **READ-ONLY** — no CREATE, INSERT, UPDATE, or DROP required.

### EXERCISE 1 | SHOW WAREHOUSES

**Task:** Write a SQL cell that runs `SHOW WAREHOUSES` and returns only the `name`, `size`, and `state` columns from the result.

How many warehouses are visible to your current role?

In [ ]:
%%sql
-- YOUR CODE HERE

### EXERCISE 2 | Snowpark DataFrame from SQL

**Task:** In a Python cell, use `session.sql()` to run `SHOW WAREHOUSES` and collect the result into a Pandas DataFrame. Print only the `name` and `size` columns.

*Hint:* `pd.DataFrame(session.sql(...).collect())`

In [ ]:
# YOUR CODE HERE

### EXERCISE 3 | Temporary Table Round-Trip

**Task:**
- A) In Python, create a Pandas DataFrame with 3 columns: `wh_name` (STRING), `size` (STRING), `is_active` (BOOLEAN) and 2 rows of your choice.
- B) Write it to Snowflake as a temporary table called `TEMP_WH_INFO`.
- C) In a SQL cell, `SELECT * FROM TEMP_WH_INFO` to confirm the round-trip.

In [ ]:
# Task A & B – YOUR CODE HERE

In [ ]:
%%sql
-- Task C – YOUR CODE HERE